# Bai Shopping Brain — iteration 2 (gated)

Run All только после corrected re-eval. Notebook физически требует валидный `REEVAL_REJECTED` handoff и human-reviewed Gold. Frozen eval берётся из re-eval evidence и не меняется. Старый deterministic train сначала санитизируется от ID/semantic collisions с frozen holdout; training запускается только после `READY_FOR_ITERATION_2`. Baseline для promotion — первый обученный кандидат, а не голая base model.


In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, 'Нужен Kaggle GPU'
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/1024**3,2),'GiB')
!pip -q install -U 'transformers>=4.51,<5' 'peft>=0.15,<1' 'datasets>=3,<5' accelerate bitsandbytes sentencepiece huggingface-hub
!rm -rf /kaggle/working/tamdeshevle /kaggle/working/bai_iteration2
!git clone --depth 1 --branch main https://github.com/eneonstudio-dev/tamdeshevle.git /kaggle/working/tamdeshevle
ROOT=Path('/kaggle/working/tamdeshevle'); WORK=Path('/kaggle/working/bai_iteration2'); WORK.mkdir(parents=True)
sys.path.insert(0,str(ROOT/'teacher-lab/training'))
from reevaluate_candidate import sha256_file
REPO_SHA=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('repo:',REPO_SHA)


In [ ]:
INPUTS=Path('/kaggle/input')
handoffs=[]
for p in INPUTS.rglob('*.json'):
    try: obj=json.loads(p.read_text(encoding='utf-8'))
    except Exception: continue
    if isinstance(obj,dict) and obj.get('kind')=='bai_existing_candidate_reeval_handoff': handoffs.append((p,obj))
if len(handoffs)!=1: raise RuntimeError(f'Need exactly one corrected re-eval handoff JSON, got {len(handoffs)}')
HANDOFF_JSON,handoff=handoffs[0]
if handoff.get('status')!='REEVAL_REJECTED' or handoff.get('next_step')!='failure_review_then_iteration_2': raise RuntimeError('Iteration 2 is allowed only after validated REEVAL_REJECTED')
def zip_by_hash(expected):
    matches=[p for p in INPUTS.rglob('*.zip') if sha256_file(p)==expected]
    if len(matches)!=1: raise RuntimeError(f'Expected exactly one ZIP for hash {expected[:12]}, got {len(matches)}')
    return matches[0]
EVIDENCE_ZIP=zip_by_hash(handoff['evidence_zip_sha256']); ADAPTER_ZIP=zip_by_hash(handoff['adapter_zip_sha256'])
def approved_human_gold(path):
    try: rows=[json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]
    except Exception: return False
    if not rows: return False
    for row in rows:
        if (row.get('review') or {}).get('status')!='approved' or (row.get('privacy') or {}).get('sanitized') is not True: return False
        source_ids={str(x.get('source_id') or '') for x in ((row.get('provenance') or {}).get('sources') or []) if isinstance(x,dict)}
        if 'human_votonobay_reviewed' not in source_ids: return False
    return True
REVIEWED_GOLD=sorted(p for p in INPUTS.rglob('gold.jsonl') if approved_human_gold(p))
if not REVIEWED_GOLD: raise RuntimeError('Need at least one human-reviewed Gold JSONL input for iteration 2')
print('REEVAL HANDOFF:',HANDOFF_JSON); print('REVIEWED GOLD:',*[str(x) for x in REVIEWED_GOLD],sep='\n- ')


In [ ]:
SOURCE_INTAKE=WORK/'source-intake'
subprocess.check_call([sys.executable,str(ROOT/'teacher-lab/training/intake_reeval_handoff.py'),'--handoff-json',str(HANDOFF_JSON),'--evidence-zip',str(EVIDENCE_ZIP),'--adapter-zip',str(ADAPTER_ZIP),'--work',str(SOURCE_INTAKE)],cwd=ROOT)
source=json.loads((SOURCE_INTAKE/'intake.json').read_text(encoding='utf-8'))
if source['status']!='REEVAL_REJECTED': raise RuntimeError('Validated source is not REEVAL_REJECTED')
EVAL_GOLD=Path(source['evidence_dir'])/'eval-gold.jsonl'
SEED=WORK/'seed'; subprocess.check_call(['node',str(ROOT/'teacher-lab/training/deterministic-seed.mjs'),str(SEED)],cwd=ROOT)
PREPARED=WORK/'prepared'
prep=[sys.executable,str(ROOT/'teacher-lab/training/prepare_iteration2_dataset.py'),'--gold',str(SEED/'gold.jsonl')]
for p in REVIEWED_GOLD: prep += ['--gold',str(p)]
prep += ['--eval-gold',str(EVAL_GOLD),'--out-dir',str(PREPARED)]
subprocess.check_call(prep,cwd=ROOT)
san=json.loads((PREPARED/'sanitization-report.json').read_text(encoding='utf-8'))
print('SANITIZATION:',json.dumps({k:v for k,v in san.items() if k!='removed'},ensure_ascii=False,indent=2))


In [ ]:
READINESS_WORK=WORK/'readiness-work'; READINESS=WORK/'readiness.json'
subprocess.check_call([sys.executable,str(ROOT/'teacher-lab/training/iteration2_readiness.py'),'--reeval-handoff-json',str(HANDOFF_JSON),'--reeval-evidence-zip',str(EVIDENCE_ZIP),'--reeval-adapter-zip',str(ADAPTER_ZIP),'--aggregate-dir',str(PREPARED/'dataset'),'--work',str(READINESS_WORK),'--out',str(READINESS)],cwd=ROOT)
ready=json.loads(READINESS.read_text(encoding='utf-8'))
if ready.get('state')!='READY_FOR_ITERATION_2': raise RuntimeError('Iteration 2 readiness did not pass')
validated_source=json.loads((READINESS_WORK/'reeval-intake/intake.json').read_text(encoding='utf-8'))
BASELINE_ADAPTER=Path(validated_source['adapter_dir']); EVAL_GOLD=Path(validated_source['evidence_dir'])/'eval-gold.jsonl'
print(json.dumps(ready,ensure_ascii=False,indent=2))


In [ ]:
OUT=Path('/kaggle/working/bai_iteration2_train')
cmd=[sys.executable,str(ROOT/'teacher-lab/training/kaggle_train_pipeline.py'),'--gold',str(PREPARED/'dataset/gold.jsonl'),'--eval-gold',str(EVAL_GOLD),'--out',str(OUT),'--baseline-adapter',str(BASELINE_ADAPTER)]
train_run=subprocess.run(cmd,cwd=ROOT)
manifest_file=OUT/'pipeline-manifest.json'
if not manifest_file.is_file(): raise RuntimeError(f'Iteration 2 pipeline failed before manifest, exit={train_run.returncode}')
pipeline=json.loads(manifest_file.read_text(encoding='utf-8'))
if train_run.returncode!=0 and pipeline.get('status')!='REJECTED': raise RuntimeError(f'Unexpected training failure exit={train_run.returncode} status={pipeline.get("status")}')
print('ITERATION 2 STATUS:',pipeline.get('status'))


In [ ]:
COMPARISON=OUT/'comparison'
required=[OUT/'baseline-predictions.jsonl',OUT/'candidate-predictions.jsonl',OUT/'metrics/baseline.json',OUT/'metrics/candidate.json']
if all(p.is_file() for p in required):
    subprocess.check_call([sys.executable,str(ROOT/'teacher-lab/training/compare_candidate_runs.py'),'--eval-gold',str(EVAL_GOLD),'--run','iteration1-baseline',str(OUT/'baseline-predictions.jsonl'),str(OUT/'metrics/baseline.json'),'--run','iteration2-candidate',str(OUT/'candidate-predictions.jsonl'),str(OUT/'metrics/candidate.json'),'--out-dir',str(COMPARISON)],cwd=ROOT)
    print((COMPARISON/'candidate-comparison.md').read_text(encoding='utf-8'))
if pipeline.get('status')=='REJECTED' and (OUT/'candidate-predictions.jsonl').is_file():
    FAIL=OUT/'failure-analysis'
    subprocess.check_call([sys.executable,str(ROOT/'teacher-lab/training/analyze_candidate_failures.py'),'--eval-gold',str(EVAL_GOLD),'--predictions',str(OUT/'candidate-predictions.jsonl'),'--out-dir',str(FAIL)],cwd=ROOT)
summary={'schema_version':'1.0','iteration':2,'state':pipeline.get('status'),'repo_sha':REPO_SHA,'baseline_candidate_sha256':ready['candidate_adapter_sha256'],'eval_gold_sha256':ready['eval_gold_sha256'],'clean_gold_sha256':ready['gold_sha256'],'examples':ready['examples'],'holdout_id_overlap':ready['holdout_id_overlap'],'holdout_fingerprint_overlap':ready['holdout_fingerprint_overlap'],'promotion_gate_unchanged':True,'release_created':bool(pipeline.get('brain_release_manifest'))}
(OUT/'iteration2-summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
bundle=shutil.make_archive('/kaggle/working/bai_iteration2_result','zip',OUT)
print(json.dumps(summary,ensure_ascii=False,indent=2)); print('RESULT ZIP:',bundle)
if pipeline.get('status')=='REJECTED': print('Candidate rejected safely; review failure-analysis before any iteration 3.')
elif pipeline.get('status')=='PROMOTION_READY': print('Candidate passed promotion; use staged release review, not automatic activation.')
